# Overview

This figure shows the performance of the best-performing elevation →
$D/K$ model as a predicted-vs-true scatter with KDE density contours.
:scope: paper :figure: 1

## Data source

``` example
analysis/all_test_performance.csv
```

For the plot

``` example
analysis/overall_performance.csv
```

For selecting run

# Setup

``` python
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sqlite3
import seaborn as sns
from matplotlib.lines import Line2D
from neural_spd.config import PROJECT_ROOT
from neural_spd import plot_styles
from neural_spd.plot_styles import cm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
plot_styles.apply()
PERF_METRICS_PATH = PROJECT_ROOT / "analysis/overall_performance.csv"
RAW_PERF_PATH = PROJECT_ROOT / "analysis/all_test_performance.csv"
```

# Load and select data

## Performance data

``` python
perf_metrics_df = pd.read_csv(PERF_METRICS_PATH)
best_perf_row = perf_metrics_df[
    (perf_metrics_df["target"] == "DoK") &
    (perf_metrics_df["data"] == "elevation") &
    (perf_metrics_df["noise"] == 0)
].sort_values("nrmse").iloc[0]
best_seed = best_perf_row["seed"]
raw_perf_df = pd.read_csv(RAW_PERF_PATH)
best_raw_perf_df = raw_perf_df[
    (raw_perf_df["seed"] == best_seed) &
    (raw_perf_df["target"] == "DoK") &
    (raw_perf_df["data"] == "elevation") &
    (raw_perf_df["noise"] == 0)
]
```

## Hillshade

``` python
def hillshade(z, azimuth=315.0, angle_altitude=45.0):
    """Generate a hillshade image from DEM.

    Notes: adapted by G. Tucker from example on GeoExamples blog,
    published March 24, 2014, by Roger Veciana i Rovira.
    """
    x, y = np.gradient(z)
    slope = np.pi / 2.0 - np.arctan(np.sqrt(x**2 + y**2))  # slope gradient
    aspect = np.arctan2(-x, y)  # aspect
    azimuthrad = azimuth * np.pi / 180.0  # convert lighting azimuth to radians
    altituderad = angle_altitude * np.pi / 180.0  # convert lighting altitude to radians
    shaded = np.sin(altituderad) * np.sin(slope) + np.cos(altituderad) * np.cos(
        slope
    ) * np.cos(azimuthrad - aspect)
    return 255 * (shaded + 1) / 2  # return result scaled 0 to 255

# SELECT RUNS
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("SELECT \"model_param.diffuser.D\"/\"model_param.streampower.k\", model_run_id FROM model_run_params")
runs = cursor.fetchall()
runs.sort(key=lambda x: x[0])
# select 10 and 90th percentile runs
low_run = runs[int(len(runs) * 0.1)]
high_run = runs[int(len(runs) * 0.9)]
array_path = PROJECT_ROOT / "data" / "0" / "elevation"
low_ar = np.load(array_path /f"{low_run[1]}.npy")
high_ar = np.load(array_path /f"{high_run[1]}.npy")

low_hsh = hillshade(low_ar)
high_hsh = hillshade(high_ar)

```

# Plotting function

``` python

def plot_raw_perf(ax):
    def add_landscape_inset(hsh, pos_rect, label, sub_label):
        ax_card = ax.inset_axes(pos_rect)
        ax_card.set_facecolor('#e6d9ad') 
        ax_card.set_xticks([])
        ax_card.set_yticks([])
        ax_inset = ax_card.inset_axes([0.1, 0.1, 0.8, 0.75])
        ax_inset.set_facecolor('#e6d9ad')
        ax_inset.imshow(hsh, cmap="gray", aspect="equal")#, extent=(0, 100, 0, 300)) 
#        ax_inset.set_xlim(0, 1)
#        ax_inset.set_ylim(0, 1)
        #ax_inset.set_title(label, fontsize=plt.rcParams["axes.titlesize"])
        ax_inset.axis("off")
        ax_card.set_xticks([]); ax_card.set_yticks([])
        ax_card.text(0.05, 1.05, label, transform=ax_inset.transAxes, 
                fontsize=9, fontweight='bold', va='bottom')
#        ax_card.axis("off")
        ax_card.spines[:].set_visible(False)
        #for spine in ax_card.spines.values():
        #    spine.set_linewidth(0.5)
        #    spine.set_color('gray')
        ax_card.text(0.5, -0.05, sub_label, transform=ax_inset.transAxes, 
                fontsize=9, ha='center', va='top')

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("True $D/K$")
    ax.set_ylabel("Predicted $D/K$")

    # Scatter of individual predictions (subtle, behind KDE)
    ax.scatter(
        best_raw_perf_df["true_labels"],
        best_raw_perf_df["predictions"],
        s=3, alpha=0.15, color=plot_styles.COLORS["gray_mid"],
        edgecolor="none", rasterized=True, zorder=1,
    )

    # KDE density contours
    sns.kdeplot(
        data=best_raw_perf_df,
        x="true_labels",
        y="predictions",
        ax=ax,
        fill=True,
        alpha=0.7,
        levels=6,
        cmap="pub_greens",
        zorder=2,
    )

    # 1:1 reference line
    lo = min(best_raw_perf_df["true_labels"].min(),
             best_raw_perf_df["predictions"].min())
    hi = max(best_raw_perf_df["true_labels"].max(),
             best_raw_perf_df["predictions"].max())
    ax.plot(
        [lo, hi], [lo, hi],
        color=plot_styles.COLORS["black"],
        linestyle="--",
        linewidth=0.8,
        zorder=3,
        label="1 : 1",
    )

    # NRMSE annotation
    nrmse = best_perf_row["nrmse"]
    ax.text(
        0.97, 0.03, f"NRMSE = {nrmse:.4f}",
        transform=ax.transAxes, ha="right", va="bottom",
        fontsize=plt.rcParams["legend.fontsize"],
        color=plot_styles.COLORS["gray_dark"],
    )

    # Legend
    legend_elements = [
        Line2D([0], [0], marker="o", color="none",
               markerfacecolor=plot_styles.COLORS["gray_mid"],
               markersize=3, alpha=0.5, label="Test samples"),
        Line2D([0], [0], marker="s", color="none",
               markerfacecolor=plot_styles.COLORS["green_dark"],
               markersize=5, label="KDE density"),
        Line2D([0], [0], color=plot_styles.COLORS["black"],
               linestyle="--", linewidth=0.8, label="1 : 1"),
    ]
    add_landscape_inset(low_hsh, [0.0,0.0,0.15,0.4], "(b)",  "Low D/K")
    add_landscape_inset(high_hsh, [0.85,0.0,0.15,0.4], "(c)", "High D/K")
    ax.text(1.5, 250, "(a)", fontsize=9, fontweight="bold")
    ax.legend(handles=legend_elements, loc="upper left")
```

# Generate Plots

``` python
plot_styles.single_column()
fig, ax = plt.subplots(figsize=(17*cm, 17*cm))
plot_raw_perf(ax)
plot_styles.save_figure(fig, "performance_raw", PROJECT_ROOT / "paper" / "figs")
fig.show()
```